In [1]:
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime

STAGING_DIR = "../data/staging/"
CLEAN_DIR   = "../data/clean/"
os.makedirs(CLEAN_DIR, exist_ok=True)

# Load semua staging files
eia_gen    = pd.read_csv(f"{STAGING_DIR}eia_generation_2022_2024.csv")
eia_retail = pd.read_csv(f"{STAGING_DIR}eia_retail_2022_2024.csv")
wb_cpi     = pd.read_csv(f"{STAGING_DIR}wb_cpi_2022_2024.csv")
wb_gdp     = pd.read_csv(f"{STAGING_DIR}wb_gdp_2022_2024.csv")
wb_ind     = pd.read_csv(f"{STAGING_DIR}wb_industrial_2022_2024.csv")
wb_unemp   = pd.read_csv(f"{STAGING_DIR}wb_unemployment_2022_2024.csv")
wb_fx      = pd.read_csv(f"{STAGING_DIR}wb_exchange_rate_2022_2024.csv")

print("Semua staging files berhasil di-load ✓")
print(f"\nRingkasan:")
for name, df in [
    ("eia_generation", eia_gen), ("eia_retail", eia_retail),
    ("wb_cpi", wb_cpi), ("wb_gdp", wb_gdp), ("wb_industrial", wb_ind),
    ("wb_unemployment", wb_unemp), ("wb_exchange_rate", wb_fx)
]:
    print(f"  {name:<20} {str(df.shape):>12}")

Semua staging files berhasil di-load ✓

Ringkasan:
  eia_generation          (3324, 9)
  eia_retail              (144, 11)
  wb_cpi                   (288, 3)
  wb_gdp                   (288, 3)
  wb_industrial            (288, 3)
  wb_unemployment          (216, 3)
  wb_exchange_rate         (288, 3)


In [8]:
def transform_retail(df_raw):
    df = df_raw.copy()
    if df.empty:
        return df

    # Kolom aktual: period, stateid, stateDescription, sectorid,
    # sectorName, sales, price, revenue, sales-units, price-units, revenue-units
    print(f"Kolom asli: {df.columns.tolist()}")

    # Pilih kolom yang relevan saja, buang kolom units
    cols_keep = ["period", "stateid", "sectorid", "sectorName",
                 "sales", "price", "revenue"]
    df = df[cols_keep]

    # Rename ke snake_case konsisten
    df = df.rename(columns={
        "stateid"    : "state_id",
        "sectorid"   : "sector_id",
        "sectorName" : "sector_name",
        "sales"      : "retail_sales_mwh",
        "price"      : "price_cents_kwh",
        "revenue"    : "revenue_million_usd",
    })

    # Filter US national only
    df = df[df["state_id"] == "US"].copy()

    # Konversi numerik
    for col in ["retail_sales_mwh", "price_cents_kwh", "revenue_million_usd"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Buang baris yang keduanya NaN
    df = df.dropna(subset=["price_cents_kwh", "retail_sales_mwh"], how="all")
    df = df.drop_duplicates()

    # Normalisasi period — sudah format YYYY-MM dari EIA
    df["period"]     = pd.to_datetime(df["period"], format="%Y-%m", errors="coerce")
    df["year"]       = df["period"].dt.year
    df["month"]      = df["period"].dt.month
    df["quarter"]    = df["period"].dt.quarter
    df["period_str"] = df["period"].dt.strftime("%Y-%m")

    # Revenue estimate sebagai crosscheck
    df["revenue_estimate"] = df["retail_sales_mwh"] * df["price_cents_kwh"] / 100

    df = df.reset_index(drop=True)

    print(f"✓ EIA Retail selesai: {df.shape}")
    print(f"  Period : {df['period_str'].min()} → {df['period_str'].max()}")
    print(f"  Sectors: {df['sector_id'].unique().tolist()}")
    print(f"  Missing price: {df['price_cents_kwh'].isna().sum()}")

    return df

df_retail_clean = transform_retail(eia_retail)
df_retail_clean.head(5)

Kolom asli: ['period', 'stateid', 'stateDescription', 'sectorid', 'sectorName', 'sales', 'price', 'revenue', 'sales-units', 'price-units', 'revenue-units']
✓ EIA Retail selesai: (144, 12)
  Period : 2022-01 → 2024-12
  Sectors: ['ALL', 'COM', 'IND', 'RES']
  Missing price: 0


,period,state_id,sector_id,sector_name,retail_sales_mwh,price_cents_kwh,revenue_million_usd,year,month,quarter,period_str,revenue_estimate
0,2022-01-01,US,ALL,all sectors,338656.04633,11.24,38053.17761,2022,1,1,2022-01,38064.939607
1,2022-01-01,US,COM,commercial,113605.09055,11.26,12793.64264,2022,1,1,2022-01,12791.933196
2,2022-01-01,US,IND,industrial,83982.00591,7.19,6037.27387,2022,1,1,2022-01,6038.306225
3,2022-01-01,US,RES,residential,140504.06917,13.64,19162.69980,2022,1,1,2022-01,19164.755035
4,2022-02-01,US,ALL,all sectors,305863.07052,11.42,34929.48889,2022,2,1,2022-02,34929.562653


In [9]:
def transform_generation(df_raw):
    df = df_raw.copy()
    if df.empty:
        return df

    # Kolom aktual: period, location, stateDescription, sectorid,
    # sectorDescription, fueltypeid, fuelTypeDescription,
    # generation, generation-units
    print(f"Kolom asli: {df.columns.tolist()}")

    # Pilih kolom relevan, buang units dan description yang redundan
    cols_keep = ["period", "location", "sectorid",
                 "fueltypeid", "fuelTypeDescription", "generation"]
    df = df[cols_keep]

    # Rename
    df = df.rename(columns={
        "location"           : "location_id",
        "sectorid"           : "sector_id",
        "fueltypeid"         : "fuel_type_id",
        "fuelTypeDescription": "fuel_description",
        "generation"         : "net_generation_mwh",
    })

    # Konversi numerik
    df["net_generation_mwh"] = pd.to_numeric(df["net_generation_mwh"], errors="coerce")
    df = df.dropna(subset=["net_generation_mwh"])
    df = df[df["net_generation_mwh"] >= 0]  # buang nilai negatif anomali

    # Normalisasi period — sudah format YYYY-MM dari EIA
    df["period"]     = pd.to_datetime(df["period"], format="%Y-%m", errors="coerce")
    df["year"]       = df["period"].dt.year
    df["month"]      = df["period"].dt.month
    df["quarter"]    = df["period"].dt.quarter
    df["period_str"] = df["period"].dt.strftime("%Y-%m")

    # Fuel mapping lengkap berdasarkan data aktual
    fuel_mapping = {
        "COL": "Coal",
        "NG" : "Natural Gas",
        "NUC": "Nuclear",
        "WND": "Wind",
        "SUN": "Solar",
        "HYC": "Hydroelectric",
        "GEO": "Geothermal",
        "OTH": "Other",
        "PET": "Petroleum",
        "OOG": "Other Gas",
        "WAS": "Waste",
        "BIO": "Biomass",
        "TSN": "All Sources",
    }
    df["fuel_name"] = df["fuel_type_id"].map(fuel_mapping).fillna("Other")

    # Kategori bahan bakar
    fossil_fuels    = ["COL", "NG", "PET", "OOG"]
    renewable_fuels = ["WND", "SUN", "HYC", "GEO", "WAS", "BIO"]
    nuclear_fuels   = ["NUC"]

    def get_category(fid):
        if fid in fossil_fuels:    return "Fossil"
        if fid in renewable_fuels: return "Renewable"
        if fid in nuclear_fuels:   return "Nuclear"
        return "Other"

    df["fuel_category"] = df["fuel_type_id"].apply(get_category)
    df["is_renewable"]  = df["fuel_type_id"].isin(renewable_fuels)

    df = df.drop_duplicates()
    df = df.reset_index(drop=True)

    print(f"✓ EIA Generation selesai: {df.shape}")
    print(f"  Period  : {df['period_str'].min()} → {df['period_str'].max()}")
    print(f"  Fuels   : {df['fuel_type_id'].unique().tolist()}")
    print(f"  Category: {df['fuel_category'].value_counts().to_dict()}")

    return df

df_gen_clean = transform_generation(eia_gen)
df_gen_clean.head(5)

Kolom asli: ['period', 'location', 'stateDescription', 'sectorid', 'sectorDescription', 'fueltypeid', 'fuelTypeDescription', 'generation', 'generation-units']
✓ EIA Generation selesai: (3312, 13)
  Period  : 2022-01 → 2024-12
  Fuels   : ['COL', 'GEO', 'HYC', 'NG', 'NUC', 'OTH', 'SUN', 'WND']
  Category: {'Renewable': 1584, 'Fossil': 1008, 'Other': 504, 'Nuclear': 216}


,period,location_id,sector_id,fuel_type_id,fuel_description,net_generation_mwh,year,month,quarter,period_str,fuel_name,fuel_category,is_renewable
0,2022-01-01,US,1,COL,"coal, excluding waste coal",63817.58413,2022,1,1,2022-01,Coal,Fossil,False
1,2022-01-01,US,1,GEO,geothermal,95.99719,2022,1,1,2022-01,Geothermal,Renewable,True
2,2022-01-01,US,1,HYC,conventional hydroelectric,22394.80501,2022,1,1,2022-01,Hydroelectric,Renewable,True
3,2022-01-01,US,1,NG,natural gas,66875.19976,2022,1,1,2022-01,Natural Gas,Fossil,False
4,2022-01-01,US,1,NUC,nuclear,39294.81700,2022,1,1,2022-01,Nuclear,Nuclear,False


In [10]:
def normalize_period(p):
    """Normalisasi semua format periode ke YYYY-MM."""
    p = str(p).strip()
    if "M" in p:
        # Format: 2022M01 → 2022-01
        parts = p.split("M")
        return f"{parts[0]}-{parts[1].zfill(2)}"
    if len(p) == 7 and "-" in p:
        # Sudah format 2022-01
        return p
    if len(p) == 4 and p.isdigit():
        # Format tahunan: 2022
        return p
    return None


def transform_worldbank(df_raw, indicator_name, is_annual=False):
    """
    Transform World Bank staging file.
    Kolom aktual: period_raw, country, {indicator_name}
    Format period_raw: 2022M01 (monthly) atau 2022-01 (sudah normal)
    """
    df = df_raw.copy()

    if df.empty:
        print(f"⚠ {indicator_name}: kosong")
        return pd.DataFrame()

    # Kolom aktual sudah dalam format long — period_raw, country, nilai
    print(f"Kolom asli: {df.columns.tolist()}")
    print(f"Sample period_raw: {df['period_raw'].iloc[0]}")

    # Normalisasi period
    df["period"] = df["period_raw"].apply(normalize_period)
    df = df.dropna(subset=["period"])

    if is_annual:
        # GDP: expand tahunan ke bulanan
        rows = []
        for _, row in df.iterrows():
            year = str(row["period"])
            if len(year) == 4 and year.isdigit():
                for month in range(1, 13):
                    rows.append({
                        "period"       : f"{year}-{month:02d}",
                        "country"      : row["country"],
                        indicator_name : row[indicator_name],
                    })
        df_result = pd.DataFrame(rows)
    else:
        df_result = df[["period", "country", indicator_name]].copy()

    # Konversi numerik
    df_result[indicator_name] = pd.to_numeric(
        df_result[indicator_name], errors="coerce"
    )
    df_result = df_result.dropna(subset=[indicator_name])

    # Filter 2022-2024
    df_result = df_result[
        df_result["period"].str.startswith(("2022", "2023", "2024"))
    ]

    # Tambah kolom waktu
    df_result["period_dt"] = pd.to_datetime(
        df_result["period"], format="%Y-%m", errors="coerce"
    )
    df_result["year"]    = df_result["period_dt"].dt.year
    df_result["month"]   = df_result["period_dt"].dt.month
    df_result["quarter"] = df_result["period_dt"].dt.quarter

    df_result = df_result.drop_duplicates(subset=["period", "country"])
    df_result = df_result.sort_values(["country", "period"]).reset_index(drop=True)

    print(f"✓ {indicator_name}: {df_result.shape}")
    print(f"  Period : {df_result['period'].min()} → {df_result['period'].max()}")
    print(f"  Negara : {sorted(df_result['country'].unique())}")
    print(f"  Missing: {df_result[indicator_name].isna().sum()}")

    return df_result


# Jalankan semua
print("=" * 55)
print("TRANSFORM — World Bank Indicators")
print("=" * 55)

df_cpi   = transform_worldbank(wb_cpi,   "cpi_pct_yoy",           is_annual=False)
df_gdp   = transform_worldbank(wb_gdp,   "gdp_current_usd_mn",    is_annual=False)
df_ind   = transform_worldbank(wb_ind,   "industrial_prod_usd",   is_annual=False)
df_unemp = transform_worldbank(wb_unemp, "unemployment_rate",     is_annual=False)
df_fx    = transform_worldbank(wb_fx,    "exchange_rate_lcu_usd", is_annual=False)

TRANSFORM — World Bank Indicators
Kolom asli: ['period_raw', 'country', 'cpi_pct_yoy']
Sample period_raw: 2022M01
✓ cpi_pct_yoy: (288, 7)
  Period : 2022-01 → 2024-12
  Negara : ['Brazil', 'China', 'France', 'Germany', 'India', 'Japan', 'United Kingdom', 'United States']
  Missing: 0
Kolom asli: ['period_raw', 'country', 'gdp_current_usd_mn']
Sample period_raw: 2022M01
✓ gdp_current_usd_mn: (288, 7)
  Period : 2022-01 → 2024-12
  Negara : ['Brazil', 'China', 'France', 'Germany', 'India', 'Japan', 'United Kingdom', 'United States']
  Missing: 0
Kolom asli: ['period_raw', 'country', 'industrial_prod_usd']
Sample period_raw: 2022M01
✓ industrial_prod_usd: (288, 7)
  Period : 2022-01 → 2024-12
  Negara : ['Brazil', 'China', 'France', 'Germany', 'India', 'Japan', 'United Kingdom', 'United States']
  Missing: 0
Kolom asli: ['period_raw', 'country', 'unemployment_rate']
Sample period_raw: 2022M01
✓ unemployment_rate: (216, 7)
  Period : 2022-01 → 2024-12
  Negara : ['Brazil', 'France', 'Germa

In [11]:
print("Menggabungkan semua World Bank indicators...")

# Merge semua berdasarkan period + country
df_wb_all = df_cpi[["period", "country", "cpi_pct_yoy"]].copy()

for df_other, col in [
    (df_gdp,   "gdp_current_usd_mn"),
    (df_ind,   "industrial_prod_usd"),
    (df_unemp, "unemployment_rate"),
    (df_fx,    "exchange_rate_lcu_usd"),
]:
    df_wb_all = df_wb_all.merge(
        df_other[["period", "country", col]],
        on  = ["period", "country"],
        how = "left"
    )

# Tambah kolom waktu
df_wb_all["period_dt"] = pd.to_datetime(df_wb_all["period"], format="%Y-%m")
df_wb_all["year"]      = df_wb_all["period_dt"].dt.year
df_wb_all["month"]     = df_wb_all["period_dt"].dt.month
df_wb_all["quarter"]   = df_wb_all["period_dt"].dt.quarter

print(f"✓ World Bank combined: {df_wb_all.shape}")
print(f"  Kolom    : {df_wb_all.columns.tolist()}")
print(f"  Negara   : {sorted(df_wb_all['country'].unique())}")
print(f"\nSample:")
print(df_wb_all[df_wb_all["country"] == "United States"].head(5).to_string())

Menggabungkan semua World Bank indicators...
✓ World Bank combined: (288, 11)
  Kolom    : ['period', 'country', 'cpi_pct_yoy', 'gdp_current_usd_mn', 'industrial_prod_usd', 'unemployment_rate', 'exchange_rate_lcu_usd', 'period_dt', 'year', 'month', 'quarter']
  Negara   : ['Brazil', 'China', 'France', 'Germany', 'India', 'Japan', 'United Kingdom', 'United States']

Sample:
      period        country  cpi_pct_yoy  gdp_current_usd_mn  industrial_prod_usd  unemployment_rate  exchange_rate_lcu_usd  period_dt  year  month  quarter
252  2022-01  United States     7.558806          26054600.0         2.720000e+11                4.0                    1.0 2022-01-01  2022      1        1
253  2022-02  United States     7.937279          26054600.0         2.730000e+11                3.9                    1.0 2022-02-01  2022      2        1
254  2022-03  United States     8.572205          26054600.0         2.750000e+11                3.7                    1.0 2022-03-01  2022      3      

In [12]:
print("=" * 55)
print("MEMBUAT DIMENSION TABLES")
print("=" * 55)

# ── dim_time ──────────────────────────────────
periods = pd.date_range(start="2022-01", end="2024-12", freq="MS")
dim_time = pd.DataFrame({
    "time_id"    : range(1, len(periods) + 1),
    "period"     : periods.strftime("%Y-%m"),
    "year"       : periods.year,
    "quarter"    : periods.quarter,
    "month_num"  : periods.month,
    "month_name" : periods.strftime("%B"),
    "quarter_label": ["Q" + str(q) for q in periods.quarter],
    "season"     : periods.month.map({
        12:"Winter", 1:"Winter", 2:"Winter",
        3:"Spring",  4:"Spring", 5:"Spring",
        6:"Summer",  7:"Summer", 8:"Summer",
        9:"Fall",    10:"Fall",  11:"Fall"
    }),
})
print(f"✓ dim_time: {dim_time.shape}")
print(dim_time.head(5).to_string())

# ── dim_fuel_type ─────────────────────────────
dim_fuel_type = pd.DataFrame([
    {"fuel_type_id": 1, "fuel_code": "COL", "fuel_name": "Coal",
     "category": "Fossil",    "is_renewable": False},
    {"fuel_type_id": 2, "fuel_code": "NG",  "fuel_name": "Natural Gas",
     "category": "Fossil",    "is_renewable": False},
    {"fuel_type_id": 3, "fuel_code": "NUC", "fuel_name": "Nuclear",
     "category": "Nuclear",   "is_renewable": False},
    {"fuel_type_id": 4, "fuel_code": "WND", "fuel_name": "Wind",
     "category": "Renewable", "is_renewable": True},
    {"fuel_type_id": 5, "fuel_code": "SUN", "fuel_name": "Solar",
     "category": "Renewable", "is_renewable": True},
    {"fuel_type_id": 6, "fuel_code": "HYC", "fuel_name": "Hydroelectric",
     "category": "Renewable", "is_renewable": True},
    {"fuel_type_id": 7, "fuel_code": "GEO", "fuel_name": "Geothermal",
     "category": "Renewable", "is_renewable": True},
    {"fuel_type_id": 8, "fuel_code": "OTH", "fuel_name": "Other",
     "category": "Other",     "is_renewable": False},
])
print(f"\n✓ dim_fuel_type: {dim_fuel_type.shape}")
print(dim_fuel_type.to_string())

# ── dim_sector ────────────────────────────────
dim_sector = pd.DataFrame([
    {"sector_id": 1, "sector_code": "RES", "sector_name": "Residential"},
    {"sector_id": 2, "sector_code": "COM", "sector_name": "Commercial"},
    {"sector_id": 3, "sector_code": "IND", "sector_name": "Industrial"},
    {"sector_id": 4, "sector_code": "ALL", "sector_name": "All Sectors"},
    {"sector_id": 5, "sector_code": "OTH", "sector_name": "Other"},
])
print(f"\n✓ dim_sector: {dim_sector.shape}")
print(dim_sector.to_string())

# ── dim_country ───────────────────────────────
dim_country = pd.DataFrame([
    {"country_id": 1, "country_code": "USA", "country_name": "United States",
     "region": "North America",  "income_group": "High income"},
    {"country_id": 2, "country_code": "CHN", "country_name": "China",
     "region": "East Asia",      "income_group": "Upper middle income"},
    {"country_id": 3, "country_code": "DEU", "country_name": "Germany",
     "region": "Europe",         "income_group": "High income"},
    {"country_id": 4, "country_code": "JPN", "country_name": "Japan",
     "region": "East Asia",      "income_group": "High income"},
    {"country_id": 5, "country_code": "IND", "country_name": "India",
     "region": "South Asia",     "income_group": "Lower middle income"},
    {"country_id": 6, "country_code": "GBR", "country_name": "United Kingdom",
     "region": "Europe",         "income_group": "High income"},
    {"country_id": 7, "country_code": "FRA", "country_name": "France",
     "region": "Europe",         "income_group": "High income"},
    {"country_id": 8, "country_code": "BRA", "country_name": "Brazil",
     "region": "Latin America",  "income_group": "Upper middle income"},
])
print(f"\n✓ dim_country: {dim_country.shape}")
print(dim_country.to_string())

MEMBUAT DIMENSION TABLES
✓ dim_time: (36, 8)
   time_id   period  year  quarter  month_num month_name quarter_label  season
0        1  2022-01  2022        1          1    January            Q1  Winter
1        2  2022-02  2022        1          2   February            Q1  Winter
2        3  2022-03  2022        1          3      March            Q1  Spring
3        4  2022-04  2022        2          4      April            Q2  Spring
4        5  2022-05  2022        2          5        May            Q2  Spring

✓ dim_fuel_type: (8, 5)
   fuel_type_id fuel_code      fuel_name   category  is_renewable
0             1       COL           Coal     Fossil         False
1             2        NG    Natural Gas     Fossil         False
2             3       NUC        Nuclear    Nuclear         False
3             4       WND           Wind  Renewable          True
4             5       SUN          Solar  Renewable          True
5             6       HYC  Hydroelectric  Renewable         

In [13]:
print("=" * 55)
print("MEMBUAT FACT TABLE")
print("=" * 55)

# ── Siapkan EIA retail untuk join ────────────
df_retail_us = df_retail_clean[
    df_retail_clean["sector_id"].isin(["RES", "COM", "IND", "ALL"])
][["period_str", "sector_id", "price_cents_kwh", "retail_sales_mwh", "revenue_million_usd"]].copy()
df_retail_us = df_retail_us.rename(columns={"period_str": "period"})

# ── Siapkan EIA generation untuk join ────────
df_gen_us = df_gen_clean[["period_str", "fuel_type_id",
                           "net_generation_mwh", "fuel_category", "is_renewable"]].copy()
df_gen_us = df_gen_us.rename(columns={"period_str": "period"})

# ── Join EIA retail dengan dim_time ──────────
fact = df_retail_us.merge(
    dim_time[["period", "time_id", "year", "quarter", "month_num"]],
    on="period", how="left"
)

# ── Join dengan dim_sector ───────────────────
fact = fact.merge(
    dim_sector[["sector_code", "sector_id"]].rename(
        columns={"sector_code": "sector_id", "sector_id": "sector_fk"}
    ),
    on="sector_id", how="left"
)

# ── Join dengan World Bank (US only) ─────────
wb_us = df_wb_all[df_wb_all["country"] == "United States"][
    ["period", "cpi_pct_yoy", "gdp_current_usd_mn",
     "industrial_prod_usd", "unemployment_rate", "exchange_rate_lcu_usd"]
].copy()

fact = fact.merge(wb_us, on="period", how="left")

# ── Tambah surrogate key ──────────────────────
fact = fact.reset_index(drop=True)
fact.insert(0, "fact_id", range(1, len(fact) + 1))

# ── Reorder kolom ────────────────────────────
final_cols = [
    "fact_id", "period", "time_id", "sector_id",
    "price_cents_kwh", "retail_sales_mwh", "revenue_million_usd",
    "cpi_pct_yoy", "gdp_current_usd_mn", "industrial_prod_usd",
    "unemployment_rate", "exchange_rate_lcu_usd",
]
available_final = [c for c in final_cols if c in fact.columns]
fact = fact[available_final]

print(f"✓ Fact table: {fact.shape}")
print(f"  Kolom  : {fact.columns.tolist()}")
print(f"  Period : {fact['period'].min()} → {fact['period'].max()}")
print(f"\nSample:")
print(fact.head(5).to_string())
print(f"\nMissing values:")
print(fact.isnull().sum())

MEMBUAT FACT TABLE
✓ Fact table: (144, 12)
  Kolom  : ['fact_id', 'period', 'time_id', 'sector_id', 'price_cents_kwh', 'retail_sales_mwh', 'revenue_million_usd', 'cpi_pct_yoy', 'gdp_current_usd_mn', 'industrial_prod_usd', 'unemployment_rate', 'exchange_rate_lcu_usd']
  Period : 2022-01 → 2024-12

Sample:
   fact_id   period  time_id sector_id  price_cents_kwh  retail_sales_mwh  revenue_million_usd  cpi_pct_yoy  gdp_current_usd_mn  industrial_prod_usd  unemployment_rate  exchange_rate_lcu_usd
0        1  2022-01        1       ALL            11.24      338656.04633          38053.17761     7.558806          26054600.0         2.720000e+11                4.0                    1.0
1        2  2022-01        1       COM            11.26      113605.09055          12793.64264     7.558806          26054600.0         2.720000e+11                4.0                    1.0
2        3  2022-01        1       IND             7.19       83982.00591           6037.27387     7.558806          2605

In [14]:
print("=" * 55)
print("SIMPAN KE FOLDER CLEAN")
print("=" * 55)

tables = {
    "fact_energy_economy" : fact,
    "dim_time"            : dim_time,
    "dim_fuel_type"       : dim_fuel_type,
    "dim_sector"          : dim_sector,
    "dim_country"         : dim_country,
    "eia_generation_clean": df_gen_clean,
    "worldbank_combined"  : df_wb_all,
}

for name, df in tables.items():
    out_path = f"{CLEAN_DIR}{name}.csv"
    df.to_csv(out_path, index=False)
    size = os.path.getsize(out_path)
    print(f"✓ {name:<30} {str(df.shape):>12}  ({size:,} bytes)")

print(f"\nSemua tabel tersimpan di: {CLEAN_DIR}")

SIMPAN KE FOLDER CLEAN
✓ fact_energy_economy               (144, 12)  (13,398 bytes)
✓ dim_time                            (36, 8)  (1,427 bytes)
✓ dim_fuel_type                        (8, 5)  (290 bytes)
✓ dim_sector                           (5, 3)  (122 bytes)
✓ dim_country                          (8, 5)  (383 bytes)
✓ eia_generation_clean             (3312, 13)  (278,152 bytes)
✓ worldbank_combined                (288, 11)  (24,701 bytes)

Semua tabel tersimpan di: ../data/clean/
